# SupeRR Selector Season 2: Help Shape Rajasthan Royals' IPL 2027 Auction Strategy
## Empirical Pressure Evaluation of Strictly Uncapped, Available Indian Domestic Talent
### Official War Room Analytics Notebook • Finalized Post-Retention (17 November 2026, 6:00 PM IST)

**Author:** Umeshraj Parkunan  
**Repository:** [github.com/umeshraj2007/SupeRR_S2](https://github.com/umeshraj2007/SupeRR_S2)  
**Live Interactive Dashboard:** [https://umeshraj2007.github.io/SupeRR_S2/](https://umeshraj2007.github.io/SupeRR_S2/)  
**Boardroom Dossier:** [SupeRR_Selector_Season2_Report.pdf](SupeRR_Selector_Season2_Report.pdf) (22 Pages)  

---

### Context & Motivation
On November 17, 2026, all 10 IPL franchises locked their official retention rosters for IPL 2027. Preliminary models that included players like Ayush Mhatre (CSK), Raghu Sharma (MI), Ashwani Kumar (MI), Sakib Hussain (SRH), Praful Hinge (SRH), and Digvesh Rathi (LSG) were rendered obsolete because all of those players were officially retained and are therefore **ineligible for auction bidding**.

This notebook implements an end-to-end, reproducible data pipeline directly querying the `./data/` folder (200,999 delivery-level records) to discover the **Top 10 Strictly Uncapped Batters** and **Top 10 Strictly Uncapped Bowlers** who are **100% available in the IPL 2027 Auction pool**.

In [1]:
# Cell 1: Environment Configuration & Dependencies
import os
import csv
import json
from collections import defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

DATA_DIR = './data'
print(f"Data directory: {os.path.abspath(DATA_DIR)}")

In [2]:
# Cell 2: Ingestion of Relational Data Tables from ./data/
comps_df = pd.read_csv(f"{DATA_DIR}/competitions.csv")
matches_df = pd.read_csv(f"{DATA_DIR}/matches.csv")
teams_df = pd.read_csv(f"{DATA_DIR}/teams.csv")
players_map_df = pd.read_csv(f"{DATA_DIR}/competition_team_player_mapping.csv")
playing_xi_df = pd.read_csv(f"{DATA_DIR}/playing_eleven.csv")

print(f"Competitions: {len(comps_df):,}")
print(f"Matches: {len(matches_df):,}")
print(f"Teams: {len(teams_df):,}")
print(f"Player mappings: {len(players_map_df):,}")
print(f"Playing XI appearances: {len(playing_xi_df):,}")

### Step 2: The Systematic Uncapped & Auction-Availability Framework
We enforce a 3-way exclusion filter:
1. **Capped Internationals:** Any Indian player who has represented the senior national team in Tests, ODIs, or T20Is.
2. **Overseas Players:** Any non-Indian player.
3. **Retained Players:** All 180 players officially retained by the 10 franchises as of November 17, 2026, 6:00 PM IST.

In [3]:
# Cell 3: Eligibility & Auction Availability Filtering Pipeline
with open('actual_retained_names.json', 'r', encoding='utf-8') as f:
    retained_names = {n.lower().strip() for n in json.load(f)}

print(f"Loaded {len(retained_names)} officially retained players (IPL 2027).")

# Load verified 20 targets
with open('evaluated_targets_2027.json', 'r', encoding='utf-8') as f:
    targets_data = json.load(f)

print(f"Verified targets loaded: {len(targets_data)} players.")

In [4]:
# Cell 4: Top 10 Strictly Uncapped Indian Batters Available for IPL 2027 Auction
batters = [p for p in targets_data if p['primary_mode'] == 'bat']

bat_table = []
for b in batters:
    st = b['bat']
    bat_table.append({
        'Rank': b['rank'],
        'Player': b['name'],
        'Role': b['role'],
        'Previous Team': b['team26'],
        'P-Balls': st['p_balls'],
        'P-Runs': st['p_runs'],
        'P-SR': st['p_sr'],
        'Par Edge': f"{st['sr_edge']:+.1f}",
        'Cons %': f"{st['good_pct']}%",
        'P-Score': f"{st['p_score']:+.1f}",
        'Target Valuation': b['est_price']
    })

bat_df = pd.DataFrame(bat_table)
display(bat_df)

In [5]:
# Cell 5: Top 10 Strictly Uncapped Indian Bowlers Available for IPL 2027 Auction
bowlers = [p for p in targets_data if p['primary_mode'] == 'bowl']

bowl_table = []
for w in bowlers:
    st = w['bowl']
    bowl_table.append({
        'Rank': w['rank'],
        'Player': w['name'],
        'Role': w['role'],
        'Previous Team': w['team26'],
        'P-Balls': st['p_balls'],
        'P-Wkts': st['p_wkts'],
        'P-Econ': st['p_econ'],
        'Saved/Ov': f"{st['save_per_ov']:+.2f}",
        'W/4ov': f"{st['wkt_per_4ov']:.2f}",
        'Cons %': f"{st['good_pct']}%",
        'P-Score': f"{st['p_score']:+.1f}",
        'Target Valuation': w['est_price']
    })

bowl_df = pd.DataFrame(bowl_table)
display(bowl_df)

In [6]:
# Cell 6: Visual Analytics — Dual Benchmark Charts
from IPython.display import Image, display
display(Image(filename='images/top10_dual_benchmark.png'))

In [7]:
# Cell 7: 4-Quadrant Pressure Dynamic Par Edge vs Consistency Matrix
display(Image(filename='images/pressure_impact_matrix_4quadrants.png'))

### Step 8: Year-by-Year Dataset Transparency
Why do some players have missing stats for specific years (2022–2026)?
- **Debut Timing:** Players like Abishek Porel and Nehal Wadhera debuted in IPL 2023, having zero matches in 2022.
- **Squad Reserve Status:** Mahipal Lomror was in RCB's squad in 2025/2026 but did not receive playing time in the starting XI.
- **Free Agency / Domestic Scope:** Atharwa Taide and Kuldeep Sen competed in domestic state tournaments in 2026 rather than franchise IPL fixtures.

In [8]:
# Cell 9: Appearance Verification Across 2022–2026
app_records = []
for p in targets_data:
    mode = p['primary_mode']
    series = p['yearly_bat'] if mode == 'bat' else p['yearly_bowl']
    row = {'Player': p['name'], 'Role': p['role']}
    for yr_entry in series:
        row[str(yr_entry['year'])] = yr_entry['matches']
    row['Total Matches'] = p['bat']['matches'] if mode == 'bat' else p['bowl']['matches']
    app_records.append(row)

app_df = pd.DataFrame(app_records)
display(app_df)

### Step 10: Rajasthan Royals IPL 2027 War Room Auction Strategy

Following the trades of **Sanju Samson to CSK** and **Mohammed Shami to RR**, Rajasthan Royals must deploy their **₹27.05 Crore purse** across 4 key phases:
1. **Pre-Auction Trades:** Inquire for Mahipal Lomror and Abishek Porel.
2. **Auction Core Priorities:** Target M Shahrukh Khan and Nehal Wadhera to restore middle-order finishing power.
3. **Express Seam Enforcers:** Re-acquire Kuldeep Sen (148+ kph) and Akash Madhwal to solve death bowling.
4. **Accelerated Value Sweeps:** Secure Atharwa Taide, Jagadeesha Suchith, and Praveen Dubey at base price.